Generating An Instruction Dataset via Llama 3 and Ollama

In [ ]:
from importlib.metadata import version

pkgs = [
    "tqdm",    # Progress bar（进度条工具）
]

for p in pkgs:
    print(f"{p} version: {version(p)}")  # 打印依赖版本，便于复现

Installing Ollama and Downloading Llama 3

In [ ]:
import json
import requests

# 通过本地 Ollama 的 HTTP API 调用 llama3 模型（需先安装 Ollama 并 `ollama pull llama3`）
def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat", role="user"):
    # Create the data payload as a dictionary（构造请求体）
    data = {
        "model": model,
        "seed": 123,        # for deterministic responses（固定随机种子 -> 结果可复现，这是确定性的关键）
        "temperature": 1.,   # for deterministic responses（注：真正保证复现的是上面的 seed；温度=1 只是采样强度）
        "top_p": 1,
        "messages": [
            {"role": role, "content": prompt}
        ]
    }

    # Send the POST request（流式接收：Ollama 会一行一行地返回 JSON 片段）
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()  # 非 2xx 直接抛异常
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)             # 每行是一个 JSON 片段
            if "message" in response_json:
                response_data += response_json["message"]["content"]  # 累加各片段的文本

    return response_data

result = query_model("What do Llamas eat?")  # 先测试一下 API 能否跑通
print(result)
result = query_model("What do Llamas eat?")
print(result)

In [ ]:
def extract_instruction(text):
    # 从模型输出里取第一行非空文本作为「指令」
    for content in text.split("\n"):
        if content:
            return content.strip()
# 用 Llama3 的特殊标记构造一个「空的 user 轮」提示，诱导模型以 assistant 身份「续写」出一条新指令
# （这是一种自举生成指令数据的技巧：让模型自己造 instruction）
query = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>"

result = query_model(query, role="assistant")   # 让模型扮演 assistant 生成一条指令
instruction = extract_instruction(result)
print(instruction)

In [ ]:
response = query_model(instruction, role="user")  # 再把生成的指令作为 user 输入，得到对应回答
print(response)

In [ ]:
from tqdm import tqdm

dataset_size = 5   # 演示用，只生成 5 条（instruction, output）样本
dataset = []

for i in tqdm(range(dataset_size)):

    result = query_model(query, role="assistant")   # 第一步：造指令
    instruction = extract_instruction(result)
    response = query_model(instruction, role="user")  # 第二步：为该指令生成回答
    entry = {
        "instruction": instruction,
        "output": response
    }
    dataset.append(entry)

In [ ]:
# 把自举生成的指令数据集保存为 JSON（indent=4 便于阅读）
with open("instruction-data-llama3-7b.json", "w") as file:
    json.dump(dataset, file, indent=4)
!cat instruction-data-llama3-7b.json